# Signal quality checks

- Portions of this notebook were adapted from the following MNE-NIRS examples:
  - https://mne.tools/mne-nirs/stable/auto_examples/general/plot_22_quality.html
  - https://mne.tools/mne-nirs/stable/auto_examples/general/plot_15_waveform.html

- Original MNE-NIRS example author: Robert Luke and MNE-NIRS contributors.
- Adapted and extended for the BrainHack Vanderbilt 2026 project by Haiping Huang.

- License: BSD-3-Clause. See the repository LICENSE-CODE.md

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import mne
from mne.preprocessing.nirs import (optical_density, scalp_coupling_index,  source_detector_distances, beer_lambert_law)
from mne_nirs.preprocessing import peak_power
from itertools import compress
import matplotlib.pyplot as plt
from mne_nirs.visualisation import plot_timechannel_quality_metric



The following will be generated:
- SCI values, bad pairs, proportion, by subject
  - And relevant histograms and topography
- Peak power (score, channel, time by subject)

In [ ]:
data_root = Path("D:/academic/research/fNIRS2025/data") # Make sure to change dir accordingly
QC_dir = Path("D:/academic/research/fNIRS2025/data/QC") # Make sure to change dir accordingly
subject_dir = sorted(data_root.glob("sub-*")) 
sci_threshold = 0.5
pp_threshold = 0.01
short_threshold = 0.01


def individual_QC(nirs_dir):
    subject = nirs_dir.name 
    print(f"Processing {subject}...") 
    raw = mne.io.read_raw_nirx(nirs_dir/"nirs", preload=True,verbose=False) # Because we had one more level before getting at actual recording files.
    raw_od = mne.preprocessing.nirs.optical_density(raw)
    raw_od_sci = raw_od.copy()
    sci = mne.preprocessing.nirs.scalp_coupling_index(raw_od);
    fig, ax = plt.subplots()
    ax.hist(sci);
    ax.set(xlabel = "Scalp Coupling Index (over entire recording)",
           ylabel = "Count",
           xlim = [0, 1],
           title = f"{subject} SCI distribution")
    fig.savefig(QC_dir/f"{subject}_sci_hist.png", 
                dpi = 300, 
                bbox_inches="tight")
    plt.close(fig)   
    raw_od_sci.info["bads"] = list(compress(raw_od.ch_names, sci < sci_threshold)) 
    fig_sci = raw_od_sci.plot_sensors();
    fig_sci.savefig(QC_dir/f"{subject}_sci_topo.png", 
                    dpi = 300, 
                    bbox_inches="tight")
    plt.close(fig_sci)
    raw_od_pp, scores, times = peak_power(raw_od.copy(), 
                                          time_window=10,
                                          threshold=pp_threshold) # need to think about how we store this since on different time windows.
    fig_pp = plot_timechannel_quality_metric(
    raw_od_pp,
    scores,
    times,
    threshold=pp_threshold,
    title="Peak Power Quality Evaluation",
    )
    fig_pp.set_size_inches(16, 30)
    fig_pp.savefig(QC_dir/f"{subject}_pp_topo.png", 
                   dpi = 300, 
                   bbox_inches="tight")
    plt.close(fig_pp)
    dists = source_detector_distances(raw_od.info)
    channel_pairs = [ch.rsplit(" ")[0] for ch in raw_od.ch_names]
    QC_temp = pd.DataFrame({
        "channel_pair": channel_pairs,
        "sci":sci,
        "distance_m": dists
    })
    channel_QC = (QC_temp
                  .groupby("channel_pair", as_index=False)
                  .agg({"sci": "first", "distance_m": "first"}))
    channel_QC.insert(0, "subject", subject)
    channel_QC["short"] = (channel_QC["distance_m"] < short_threshold)
    channel_QC["bad"] = (channel_QC["sci"] < sci_threshold)
    channel_QC["distance_cm"] = (channel_QC["distance_m"] * 100)
    channel_QC = channel_QC.drop(columns = "distance_m")
    n_channels = len(channel_QC)
    n_short = channel_QC["short"].sum()
    n_long = (~channel_QC["short"]).sum()
    bad_short_channels = channel_QC[
        channel_QC["bad"] & channel_QC["short"]
    ] 
    bad_long_channels = channel_QC[
    channel_QC["bad"] & ~channel_QC["short"]
    ]
    n_bad_short = (channel_QC["bad"] & channel_QC["short"]).sum()
    n_bad_long = (channel_QC["bad"] & ~channel_QC["short"]).sum()
    prop_bad_short = n_bad_short / n_short
    prop_bad_long = n_bad_long / n_long
    subject_QC = {
    "subject": subject,
    "n_channels": n_channels,
    "n_short": n_short,
    "n_long": n_long,
    "n_bad_short": n_bad_short,
    "n_bad_long": n_bad_long,
    "prop_bad_short": prop_bad_short,
    "prop_bad_long": prop_bad_long,
    "bad_short_channels" : bad_short_channels,
    "bad_long_channels" : bad_long_channels
    }


    return {"subject": subject,
            "channel_QC": channel_QC,
            "subject_QC": subject_QC,
            "raw_od_pp": raw_od_pp,
            "raw_od": raw_od
            
            }

    


- Defining the root directory for the data
- Getting a sorted list of subject directories; .glob is a pattern matching method.
  - subject_dir gives you something like "D:\academic\research\fNIRS2025\data\sub-pilot001".
- The \* is a wildcard that matches any characters, so "sub-*" will match any directory that starts with "sub-"
- .parent.name: Extracting the subject name from the directory path, so in our case: sub-pilotXXX
- Printing the subject name to indicate which subject is being processed. the f"...{}" is using a f-string.
  - The f informs Python that there is a code in the text string, in our case, 'subject'.
- When we read nirx files, it would be like using mne.io.read_raw_nirx(Path("D:/academic/research/fNIRS2025/data/sub-pilot005/nirs"))
  - mne.io.read_raw_nirx expects the destination of recording files
- The sci variable includes a histogram/count of channel by sci values.
  - In our case we have 200 channels total, i.e., 100 channels for each wavelength. The 100 comes from 84 long channels + 16 short channels.
- .info returns a MNE INFO object type. It includes data, participant, experimenter, fs, # of channels, applied filter(s), among others.
- list(compress(raw_od.ch_names, sci < sci_threshold)): 
  - Within the compress function, sci values of every channel selected was compared to the sci threshold, and returns boolean array of Trues and Falses.
  - Then, compress() returns the channel pair and relevant wavelength associated with 'True's.
  - Finally, list() wraps it up and pass to the "bads" field in the info object.
- The peak power function gives us scores by time by channel.
- ch.rsplit(" ")[0] split the text string from the right side, S1_D2 760 becomes S1_D2 AND 760, then we get the first value, S1_D2.

## Group level

In [ ]:
channel_QC = []
subject_QC = []

for nirs_dir in subject_dir:

    result = individual_QC(nirs_dir)
    channel_QC.append(result["channel_QC"])
    subject_QC.append(result["subject_QC"])

    channel_QC_df = pd.concat(channel_QC, ignore_index=True)
    subject_QC_df = pd.DataFrame(subject_QC)

## Visualize

In [ ]:
# Build a group level summary
raw_od = result["raw_od"] # All subjects had same montage, so we can just use the last subject's raw_od to get channel positions.
group_channel_QC = (channel_QC_df
                    .groupby("channel_pair", as_index=False)
                    .agg(mean_sci = ("sci", "mean"),
                         median_sci = ("sci", "median"),
                         n_bad = ("bad", "sum"),
                         n_subjects = ("subject", "nunique"),
                         short = ("short", "first")))
group_channel_QC["prop_bad"] = (group_channel_QC["n_bad"] / group_channel_QC["n_subjects"])
positions = raw_od._get_channel_positions() 
position_df = pd.DataFrame({"channel_name": raw_od.ch_names,
                            "x": positions[:, 0],
                            "y": positions[:, 1],
                            "z": positions[:, 2]})
position_df["channel_pair"] = [
    ch.rsplit(" ")[0]
    for ch in position_df["channel_name"]
]
position_df = (
    position_df
    .groupby("channel_pair", as_index=False)
    .agg({
        "x": "first",
        "y": "first",
        "z": "first"
    })
)
group_channel_QC = group_channel_QC.merge(
    position_df,
    on="channel_pair",
    how="left"
)

group_long_QC = group_channel_QC[
    ~group_channel_QC["short"]
].copy()

n_total_subjects = subject_QC_df["subject"].nunique()

fig, ax = plt.subplots(figsize=(14, 12))

sc = ax.scatter(
    group_long_QC["x"],
    group_long_QC["y"],
    c=group_long_QC["n_bad"],
    s=160,
    vmin=0,
    vmax=n_total_subjects,
    cmap = "gist_gray"
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Proportion bad across subjects")

ax.set_title("Long-channel bad frequency")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")

fig.savefig(
    QC_dir / "group_long_prop_bad_scatter.png",
    dpi=300,
    bbox_inches="tight"
)